In [22]:
import numpy as np
import matplotlib.pyplot as plt

In [23]:
# Constantes radar
c = 3e8
B = 10e6
T = 10e-6
fs = 100e6

# Fenêtre de réception
n_rx = 4000
t = np.arange(n_rx) / fs

# Distance de la cible et rapport signal/bruit
R = 3000
snr_db = 10

# Temps aller-retour de l'onde
tau = 2 * R / c

In [24]:
def chirp(t):
    return np.exp(1j * np.pi* (B/T) * t**2)

In [25]:
# Conversion du retard en nombre d'échantillons
delay_samples = round(tau * fs)

# Chirp émis
n_pulse = round(T * fs)
t_pulse = np.arange(n_pulse) / fs
tx = chirp(t_pulse)

# Echo retardé
echo = np.zeros(n_rx, dtype=complex)

start = delay_samples
stop = start + n_pulse

if stop > n_rx:
    raise ValueError("La fenêtre de réception est trop courte pour contenir l'écho.")

echo[start:stop] = tx

# Puissance du signal utile
signal_power = np.mean(np.abs(echo[start:stop]) ** 2)

# Puissance de bruit correspondant à la SNR demandée
noise_power = signal_power / 10 ** (snr_db / 10)
sigma = np.sqrt(noise_power)

# Bruit complexe gaussien
noise = sigma * (
    np.random.randn(n_rx) + 1j * np.random.randn(n_rx)
) / np.sqrt(2)

# Signal effectivement reçu
rx = echo + noise

print(f"Distance réelle       : {R:.1f} m")
print(f"Retard aller-retour   : {tau * 1e6:.2f} µs")
print(f"Retard en échantillons: {delay_samples}")
print(f"Écart-type du bruit   : {sigma:.3f}")

Distance réelle       : 3000.0 m
Retard aller-retour   : 20.00 µs
Retard en échantillons: 2000
Écart-type du bruit   : 0.316


In [26]:
ref = tx

correlation = np.correlate(rx, ref, mode="full")
out = np.abs(correlation)

# Indices correspondant aux retards positifs
lags = np.arange(len(out)) - (len(ref) - 1)

positive = lags >= 0

positive_lags = lags[positive]
positive_out = out[positive]

# Position du maximum
peak_index = np.argmax(positive_out)
estimated_delay_samples = positive_lags[peak_index]

# Conversion en temps puis en distance
estimated_tau = estimated_delay_samples / fs
estimated_R = c * estimated_tau / 2

print(f"Retard estimé   : {estimated_tau * 1e6:.2f} µs")
print(f"Distance estimée: {estimated_R:.2f} m")

Retard estimé   : 20.00 µs
Distance estimée: 3000.00 m
